In [1]:
!gdown 1CsWWnSZqO2MkNMfwKcOeTyJ4TEciCxfJ -O data/materias_embeddings.parquet

Downloading...
From: https://drive.google.com/uc?id=1CsWWnSZqO2MkNMfwKcOeTyJ4TEciCxfJ
To: /Users/camilarojasguajardo/Desktop/Magister/lobby-recsys/data/materias_embeddings.parquet
100%|██████████████████████████████████████| 36.9M/36.9M [00:00<00:00, 56.8MB/s]


In [2]:
import numpy as np
import pandas as pd
from scipy import sparse
from lightfm import LightFM
from lightfm.data import Dataset
from sklearn.preprocessing import MultiLabelBinarizer

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


# Data loading

In [3]:
materias_emb = pd.read_parquet("data/materias_embeddings.parquet")

aud = pd.read_csv(
    "data/audiencies.csv",
    usecols=["audiencia_id", "sujeto_pasivo_id", "institucion_id", "materias_tratadas", "fecha"]
)
act = pd.read_csv(
    "data/active_subjects.csv",
    usecols=["audiencia_id", "Nombre completo", "Representa a", "Calidad"]
)
pas = pd.read_csv(
    "data/passive_subjects.csv",
    usecols=["id", "nombre", "cargo", "institution_id"]
).rename(columns={"id": "sujeto_pasivo_id"})

In [4]:
import hashlib

def _norm(s):
    return (str(s) if pd.notna(s) else "").strip().lower()

def make_hash_id(s):
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16) % (10 ** 9)

In [5]:
act["user_id"] = (
    act["Nombre completo"].map(_norm) + "|" +
    act["Representa a"].map(_norm) + "|" +
    act["Calidad"].map(_norm)
)

act["user_id"] = act["user_id"].apply(make_hash_id)

# User and items features

índices consistentes

In [6]:
# Users, se crean ids numéricos para los sujetos activos
users = act.loc[:, ["user_id"]].drop_duplicates().copy()
users["u_idx"] = users["user_id"].astype("category").cat.codes

user_id_to_idx = dict(zip(users["user_id"], users["u_idx"]))
user_idx_to_id = dict(enumerate(users["user_id"]))

# Items, se crean ids numéricos para los sujetos pasivos
items = (
    aud.loc[:, ["sujeto_pasivo_id"]].drop_duplicates().rename(columns={"sujeto_pasivo_id": "item_id"})
)
items["i_idx"] = items["item_id"].astype("category").cat.codes

item_id_to_idx = dict(zip(items["item_id"], items["i_idx"]))
item_idx_to_id = dict(enumerate(items["item_id"]))


In [7]:
act

,audiencia_id,Nombre completo,Calidad,Representa a,user_id
0,691146,Leslie Zapata,Gestor de intereses,Leslie Alejandra Zapata Vásquez,991030288
1,691322,Flora Flores,Gestor de intereses,Vecinos de Caquena,672357694
2,691521,Clara Blanco,Gestor de intereses,Clara Blanco Mamani,320967789
3,692878,Conrado Blanco,Gestor de intereses,Asoc. de Ganaderos de Guallatire,944752219
4,740813,Yessica Sanches,Gestor de intereses,Consejo ADI,357461450
...,...,...,...,...,...
1153805,766862,JAIME OSORIO,Gestor de intereses,JAIME OSORIO,904505247
1153806,766864,LORENA OSORIO,Gestor de intereses,LORENA OSORIO,392918885
1153807,766866,LORENA OSORIO,Gestor de intereses,LORENA OSORIO,392918885
1153808,761890,Margarita Corrotea,Gestor de intereses,Margarita Corrotea,403358966


In [8]:
print(act["Representa a"].unique().shape)

(535189,)


In [9]:
aud

,institucion_id,sujeto_pasivo_id,fecha,audiencia_id,materias_tratadas
0,AB023,634851,2024-03-18 17:00:00-03:00,691146,"Elaboración, dictación, modificación, denegaci..."
1,AB023,634851,2024-03-21 12:30:00-03:00,691322,"Elaboración, dictación, modificación, denegaci..."
2,AB023,634851,2024-03-21 13:00:00-03:00,691521,"Elaboración, dictación, modificación, denegaci..."
3,AB023,634851,2024-03-25 11:00:00-03:00,692878,"Elaboración, dictación, modificación, denegaci..."
4,AB023,634851,2024-09-25 11:30:00-03:00,740813,"Diseño, implementación y evaluación de polític..."
...,...,...,...,...,...
632310,MU284,391700,2024-12-31 08:00:00-03:00,766862,"Diseño, implementación y evaluación de polític..."
632311,MU284,391700,2024-12-31 08:30:00-03:00,766864,"Diseño, implementación y evaluación de polític..."
632312,MU284,391700,2024-12-31 08:30:00-03:00,766866,"Diseño, implementación y evaluación de polític..."
632313,MU284,712954,2024-12-11 10:00:00-03:00,761890,"Diseño, implementación y evaluación de polític..."


In [10]:
import numpy as np
import pandas as pd

def split_userwise_holdout(
    act: pd.DataFrame,
    aud: pd.DataFrame,
    test_frac: float = 0.2,
    min_test: int = 1,
    strategy: str = "recent",  # "recent" (recomendado) o "random"
    seed: int = 42
):
    """
    Devuelve (train_aud_ids, test_aud_ids) usando un hold-out por usuario.
    - Para cada user_id, separa ~test_frac de sus audiencias (al menos min_test si tiene >=2).
    - strategy="recent": manda las más recientes al test (requiere aud.fecha; si falta, cae a random).
    - strategy="random": selección aleatoria por usuario.
    - Usuarios con 1 audiencia -> van a train (no se puede testear por usuario).
    """
    rng = np.random.default_rng(seed)

    # aseguramos fecha si se usa "recent"
    use_recent = (strategy == "recent") and ("fecha" in aud.columns)
    if use_recent:
        aud = aud.copy()
        aud["fecha"] = pd.to_datetime(aud["fecha"], errors="coerce")

    # join para saber (audiencia_id, user_id, fecha si aplica)
    df = act[["audiencia_id", "user_id"]].merge(
        aud[["audiencia_id", "fecha"]] if "fecha" in aud.columns else aud[["audiencia_id"]],
        on="audiencia_id",
        how="left"
    )

    train_ids = []
    test_ids = []

    for uid, g in df.groupby("user_id"):
        ids = g["audiencia_id"].to_numpy()
        n = len(ids)

        if n <= 1:
            # con 1 audiencia, no hacemos split (va a train)
            train_ids.extend(ids.tolist())
            continue

        # cuántas al test para este usuario
        k = max(min_test, int(np.floor(test_frac * n)))
        k = min(k, n - 1)  # siempre deja algo en train

        if use_recent and g["fecha"].notna().any():
            # ordenar por fecha asc y tomar las últimas k para test
            g2 = g.sort_values("fecha")
            test_pick = g2["audiencia_id"].tail(k).to_numpy()
        else:
            # aleatorio por usuario
            pick_idx = rng.choice(n, size=k, replace=False)
            test_pick = ids[pick_idx]

        train_pick = np.setdiff1d(ids, test_pick, assume_unique=False)

        test_ids.extend(test_pick.tolist())
        train_ids.extend(train_pick.tolist())

    # eliminar duplicados por si acaso
    train_ids = list(dict.fromkeys(train_ids))
    test_ids  = list(dict.fromkeys(test_ids))
    return train_ids, test_ids


In [11]:
def build_user_embedding_features_from_ids(
    emb_df: pd.DataFrame,
    act: pd.DataFrame,
    aud: pd.DataFrame,
    user_id_to_idx: dict,
    train_aud_ids: list,
    use_recency: bool = True,
    half_life_days: float = 180.0
) -> sparse.csr_matrix:
    tmp = (
        emb_df.loc[emb_df["audiencia_id"].isin(train_aud_ids), ["audiencia_id", "embedding"]]
        .merge(act[["audiencia_id", "user_id"]], on="audiencia_id", how="inner")
        .dropna(subset=["user_id", "embedding"])
    )

    if tmp.empty:
        n_users = (max(user_id_to_idx.values()) + 1) if user_id_to_idx else 0
        emb_dim = len(emb_df["embedding"].iloc[0]) if len(emb_df) else 0
        return sparse.csr_matrix((n_users, emb_dim), dtype="float32")

    if use_recency and ("fecha" in aud.columns):
        tmp = tmp.merge(aud[["audiencia_id", "fecha"]], on="audiencia_id", how="left")
        tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")
        tmp = tmp.loc[tmp["fecha"].notna()].copy()

        if tmp.empty:
            tmp["__w__"] = 1.0
        else:
            lam = np.log(2) / float(half_life_days)
            max_date = pd.Timestamp(tmp["fecha"].max())
            f_np = tmp["fecha"].to_numpy(dtype="datetime64[ns]")
            max_np = max_date.to_datetime64()
            ages = (max_np - f_np).astype("timedelta64[D]").astype("float32")
            ages = np.clip(ages, a_min=0, a_max=None)
            tmp["__w__"] = np.exp(-lam * ages)
    else:
        tmp["__w__"] = 1.0

    emb_dim = len(tmp["embedding"].iloc[0])
    E = np.vstack(tmp["embedding"].values).astype("float32")
    W = tmp["__w__"].to_numpy("float32")

    tmp = tmp[["user_id"]].assign(__row__=np.arange(len(tmp)))
    groups = tmp.groupby("user_id")["__row__"].apply(list)

    user_ids = groups.index.to_numpy()
    U_stack = np.zeros((len(user_ids), emb_dim), dtype="float32")
    for j, idxs in enumerate(groups.values):
        ww = W[idxs][:, None]
        vecs = E[idxs, :]
        num = (vecs * ww).sum(axis=0)
        den = ww.sum(axis=0)
        den = np.where(den == 0, 1.0, den)
        U_stack[j, :] = num / den

    # normalización L2
    norms = np.linalg.norm(U_stack, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    U_stack = U_stack / norms

    n_users = (max(user_id_to_idx.values()) + 1) if user_id_to_idx else 0
    U_mat = np.zeros((n_users, emb_dim), dtype="float32")
    u_idx_vec = pd.Series(user_ids).map(user_id_to_idx).dropna().astype(int).to_numpy()
    U_mat[u_idx_vec] = U_stack[:len(u_idx_vec), :]
    return sparse.csr_matrix(U_mat)


In [12]:
train_ids, test_ids = split_userwise_holdout(
    act=act,
    aud=aud,
    test_frac=0.2,     
    min_test=1,        
    strategy="recent",
    seed=42
)

U = build_user_embedding_features_from_ids(
    emb_df=materias_emb,    
    act=act,                
    aud=aud,                
    user_id_to_idx=user_id_to_idx,
    train_aud_ids=train_ids, 
    use_recency=True,
    half_life_days=180
)

/var/folders/pc/1tbslm8954q5q5cyk947n34h0000gn/T/ipykernel_37018/1021282315.py:25: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  aud["fecha"] = pd.to_datetime(aud["fecha"], errors="coerce")
/var/folders/pc/1tbslm8954q5q5cyk947n34h0000gn/T/ipykernel_37018/1856812066.py:23: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")


Passive subjects (users) features are the means of the embeddings of their audiencies' materias

In [13]:
def l2_normalize_rows(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms


def build_item_embedding_features_from_ids(
    emb_df: pd.DataFrame,      # dataframe con columnas [audiencia_id, embedding]
    aud: pd.DataFrame,         # dataframe con columnas [audiencia_id, sujeto_pasivo_id, fecha]
    item_id_to_idx: dict,      # mapeo item_id -> i_idx
    train_aud_ids: list,       # lista de audiencias usadas para TRAIN
    use_recency: bool = True,  # ponderar por recencia (opcional)
    half_life_days: float = 180.0
) -> sparse.csr_matrix:
    """
    Crea la matriz de features de item (autoridades/pasivos) usando embeddings
    promedio de audiencias en TRAIN, con ponderación por recencia opcional.
    """

    # --- 1) Filtrar solo audiencias del split TRAIN y unir con items ---
    tmp = (
        emb_df.loc[emb_df["audiencia_id"].isin(train_aud_ids), ["audiencia_id", "embedding"]]
        .merge(
            aud[["audiencia_id", "sujeto_pasivo_id", "fecha"]],
            on="audiencia_id", how="inner"
        )
        .rename(columns={"sujeto_pasivo_id": "item_id"})
        .dropna(subset=["item_id", "embedding"])
    )

    # --- 2) Si no hay registros válidos, devolver matriz vacía ---
    if tmp.empty:
        n_items = (max(item_id_to_idx.values()) + 1) if item_id_to_idx else 0
        emb_dim = len(emb_df["embedding"].iloc[0]) if len(emb_df) else 0
        return sparse.csr_matrix((n_items, emb_dim), dtype="float32")

    # --- 3) Calcular pesos por recencia (opcional) ---
    if use_recency and ("fecha" in tmp.columns):
        tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")
        tmp = tmp.loc[tmp["fecha"].notna()].copy()

        if tmp.empty:
            tmp["__w__"] = 1.0
        else:
            lam = np.log(2) / float(half_life_days)
            max_date = pd.Timestamp(tmp["fecha"].max())
            f_np = tmp["fecha"].to_numpy(dtype="datetime64[ns]")
            max_np = max_date.to_datetime64()
            ages = (max_np - f_np).astype("timedelta64[D]").astype("float32")
            ages = np.clip(ages, a_min=0, a_max=None)
            tmp["__w__"] = np.exp(-lam * ages)
    else:
        tmp["__w__"] = 1.0

    # --- 4) Pooling (promedio ponderado) de embeddings por item ---
    emb_dim = len(tmp["embedding"].iloc[0])
    E = np.vstack(tmp["embedding"].values).astype("float32")
    W = tmp["__w__"].to_numpy("float32")

    tmp = tmp[["item_id"]].assign(__row__=np.arange(len(tmp)))
    groups = tmp.groupby("item_id")["__row__"].apply(list)

    item_ids = groups.index.to_numpy()
    I_stack = np.zeros((len(item_ids), emb_dim), dtype="float32")

    for j, idxs in enumerate(groups.values):
        ww = W[idxs][:, None]             # (m,1)
        vecs = E[idxs, :]                 # (m,d)
        num = (vecs * ww).sum(axis=0)
        den = ww.sum(axis=0)
        den = np.where(den == 0, 1.0, den)
        I_stack[j, :] = num / den

    # --- 5) Normalización L2 por fila ---
    I_stack = l2_normalize_rows(I_stack)

    # --- 6) Alinear al orden i_idx (LightFM requiere consistencia) ---
    n_items = (max(item_id_to_idx.values()) + 1) if item_id_to_idx else 0
    I_mat = np.zeros((n_items, emb_dim), dtype="float32")
    i_idx_vec = pd.Series(item_ids).map(item_id_to_idx).dropna().astype(int).to_numpy()
    I_mat[i_idx_vec] = I_stack[:len(i_idx_vec), :]

    # --- 7) Devolver matriz dispersa ---
    return sparse.csr_matrix(I_mat)


In [14]:
I = build_item_embedding_features_from_ids(
    emb_df=materias_emb,
    aud=aud,
    item_id_to_idx=item_id_to_idx,
    train_aud_ids=train_ids,
    use_recency=True,
    half_life_days=180
)


/var/folders/pc/1tbslm8954q5q5cyk947n34h0000gn/T/ipykernel_37018/280921552.py:39: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")


# Interactions

In [23]:
from scipy import sparse
import numpy as np
import pandas as pd

def build_interactions_from_ids(
    aud, act, user_id_to_idx, item_id_to_idx, audience_ids, return_df=False
):
    # 1) join básico
    df = (
        aud.loc[aud["audiencia_id"].isin(audience_ids), ["audiencia_id","sujeto_pasivo_id"]]
        .merge(act[["audiencia_id","user_id"]], on="audiencia_id", how="inner")
    )

    # 2) mapear a índices; descartar NaN de mapping
    df["u_idx"] = df["user_id"].map(user_id_to_idx)
    df["i_idx"] = df["sujeto_pasivo_id"].map(item_id_to_idx)
    df = df.dropna(subset=["u_idx","i_idx"]).copy()

    # 3) asegurar enteros nativos (no 'Int64' nullable)
    df["u_idx"] = df["u_idx"].astype(np.int64)
    df["i_idx"] = df["i_idx"].astype(np.int64)

    # 4) eliminar duplicados exactos (u,i) y consolidar (por si hay múltiples audiencias)
    df = df.groupby(["u_idx","i_idx"], as_index=False).size()  # cuenta ocurrencias
    # si quieres binario:
    df["data"] = 1.0
    # si quisieras ponderar por recencia/veces, usa 'size' como peso:
    # df["data"] = df["size"].astype(np.float32)

    # 5) dimensiones de la matriz (basadas en mapeos)
    n_users = int(max(user_id_to_idx.values()) + 1) if user_id_to_idx else 0
    n_items = int(max(item_id_to_idx.values()) + 1) if item_id_to_idx else 0

    # 6) construir CSR
    mat = sparse.csr_matrix(
        (df["data"].to_numpy(np.float32), (df["u_idx"].to_numpy(), df["i_idx"].to_numpy())),
        shape=(n_users, n_items),
        dtype=np.float32
    )
    return (mat, df[["u_idx","i_idx"]].copy()) if return_df else mat


# --- construir y limpiar test respecto de train ---
train, train_df = build_interactions_from_ids(aud, act, user_id_to_idx, item_id_to_idx, train_ids, return_df=True)
test,  test_df  = build_interactions_from_ids(aud, act, user_id_to_idx, item_id_to_idx, test_ids,  return_df=True)

# quitar pares (u,i) que están en TRAIN
u_train = train_df["u_idx"].astype(np.int64).to_numpy()
i_train = train_df["i_idx"].astype(np.int64).to_numpy()
u_test  = test_df["u_idx"].astype(np.int64).to_numpy()
i_test  = test_df["i_idx"].astype(np.int64).to_numpy()

key_train = (u_train << 32) + i_train
key_test  = (u_test  << 32) + i_test

mask = ~np.isin(key_test, key_train)

test_df = test_df.loc[mask]

# rehacer TEST ya limpio
n_users, n_items = train.shape
test = sparse.csr_matrix(
    (np.ones(len(test_df), dtype=np.float32), (test_df["u_idx"].to_numpy(), test_df["i_idx"].to_numpy())),
    shape=(n_users, n_items),
    dtype=np.float32
)


# Trainning

In [24]:
def describe_sparse_matrix(name, mat):
    """Imprime dimensiones y densidad de una matriz CSR."""
    n_rows, n_cols = mat.shape
    nnz = mat.nnz
    density = nnz / (n_rows * n_cols) if n_rows * n_cols > 0 else 0
    print(f"{name}: {n_rows} × {n_cols}  |  nnz={nnz:,}  |  densidad={density:.6f}")

# Ejemplo de uso
describe_sparse_matrix("Interacciones TRAIN", train)
describe_sparse_matrix("Interacciones TEST", test)
describe_sparse_matrix("User features U", U)
describe_sparse_matrix("Item features I", I)


Interacciones TRAIN: 712057 × 20481  |  nnz=953,972  |  densidad=0.000065
Interacciones TEST: 712057 × 20481  |  nnz=46,612  |  densidad=0.000003
User features U: 712057 × 384  |  nnz=273,394,176  |  densidad=0.999869
Item features I: 20481 × 384  |  nnz=7,665,792  |  densidad=0.974708


In [25]:
import os, time, numpy as np
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, auc_score

# --- Configurables rápidos ---
n_epochs = 30
eval_every = 5                 # evaluar cada N épocas (imprime siempre ep 1 y final)
sample_users_max = 2000        # tamaño de la muestra para métricas
k_eval = 10                    # P@k
patience = 3                   # early stopping por P@k en la muestra
use_early_stopping = True

# Ajusta hilos según tu CPU
num_threads = min(12, (os.cpu_count() or 4))

# Modelo (puedes bajar no_components=32 para +velocidad)
model = LightFM(
    loss="warp",
    no_components=64,
    learning_rate=0.05,
    user_alpha=1e-6,
    item_alpha=1e-6,
    random_state=42
)

# --- Prepara muestra de usuarios para evaluar rápido ---
# usuarios con al menos 1 interacción en TEST
test_csr = test.tocsr()
eligible_users = np.unique(test_csr.nonzero()[0])
if eligible_users.size > 0:
    rng = np.random.default_rng(42)
    sample_size = min(sample_users_max, eligible_users.size)
    user_sample = rng.choice(eligible_users, size=sample_size, replace=False)
else:
    user_sample = None  # no sample -> eval completa (si aplica)

best_p10 = -np.inf
epochs_no_improve = 0

for epoch in range(1, n_epochs + 1):
    t0 = time.time()
    # una época, imprime pérdida interna rápido (sin métricas externas)
    model.fit_partial(
        interactions=train,
        user_features=U,
        item_features=I,
        epochs=1,
        num_threads=8,
        verbose=True  # muestra "Epoch X: loss ..."
    )
    dt = time.time() - t0

    # ¿tocan métricas?
    do_eval = (epoch == 1) or (epoch % eval_every == 0) or (epoch == n_epochs)

    if do_eval:
        # --- Evaluación sobre una muestra de usuarios ---
        if user_sample is not None:
            # crea vistas parciales de las matrices para esos usuarios
            test_sub = test[user_sample, :]
            train_sub = train[user_sample, :]
            U_sub = U[user_sample, :]

            P10 = precision_at_k(model, test_sub, train_interactions=train_sub,
                                user_features=U_sub, item_features=I,
                                k=k_eval, num_threads=num_threads).mean()
            AUC = auc_score(model, test_sub, train_interactions=train_sub,
                            user_features=U_sub, item_features=I,
                            num_threads=num_threads).mean()
        else:
            P10 = precision_at_k(model, test, train_interactions=train,
                                user_features=U, item_features=I,
                                k=k_eval, num_threads=num_threads).mean()
            AUC = auc_score(model, test, train_interactions=train,
                            user_features=U, item_features=I,
                            num_threads=num_threads).mean()


        eta = (n_epochs - epoch) * dt
        print(f"[Epoch {epoch:02d}/{n_epochs}] {dt:.2f}s | ETA~{eta/60:.1f}m | P@{k_eval}={P10:.4f} | AUC={AUC:.4f}")

        # Early stopping simple por P@k en la muestra
        if use_early_stopping:
            if P10 > best_p10 + 1e-6:
                best_p10 = P10
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"Early stopping: sin mejora en {patience} evaluaciones (mejor P@{k_eval}={best_p10:.4f}).")
                    break
    else:
        eta = (n_epochs - epoch) * dt
        print(f"[Epoch {epoch:02d}/{n_epochs}] {dt:.2f}s | ETA~{eta/60:.1f}m (sin evaluación)")


Epoch: 100%|██████████| 1/1 [03:09<00:00, 189.76s/it]


[Epoch 01/30] 189.83s | ETA~91.8m | P@10=0.0000 | AUC=0.7004


Epoch: 100%|██████████| 1/1 [03:10<00:00, 190.46s/it]


[Epoch 02/30] 190.50s | ETA~88.9m (sin evaluación)


Epoch: 100%|██████████| 1/1 [03:10<00:00, 190.18s/it]


...

[Epoch 18/30] 244.56s | ETA~48.9m (sin evaluación)


Epoch: 100%|██████████| 1/1 [04:00<00:00, 240.46s/it]


[Epoch 19/30] 240.50s | ETA~44.1m (sin evaluación)


Epoch: 100%|██████████| 1/1 [03:38<00:00, 218.21s/it]


[Epoch 20/30] 218.25s | ETA~36.4m | P@10=0.0001 | AUC=0.7045
Early stopping: sin mejora en 3 evaluaciones (mejor P@10=0.0001).
